# NARX Training

This notebook trains and evaluates a NARX neural model for the hydraulic subsystem. The trained model is used as a surrogate predictor in the turbine estimation examples.


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from copy import deepcopy


# ============================================================
# 1. CONFIG & PATH
# ============================================================
@dataclass
@dataclass
class NARXConfig:
    # -------- data --------
    seq_len: int = 200        # 1.0 s memory (dt=0.01)
    stride: int = 20
    batch_size: int = 64

    # -------- model --------
    hidden_dim: int = 128
    dropout: float = 0.05

    # -------- training --------
    lr: float = 1e-4
    weight_decay: float = 1e-6
    epochs: int = 650
    patience: int = 137

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # -------- path --------
    csv_path: str = (
        "data/hydraulic-dataset/hydraulic-dataset.csv"
    )

# ============================================================
# 2. NARX MODEL (Using Measured q as Feature)
# ============================================================
class NARXPredictor(nn.Module):
    """
    NARX MLP:
      dynamic inputs: [x_n(t-k:t), q(t-k:t)]
      static inputs : [T_e, T_q, y_r, h0]
      output         : x3(t+1)
    """
    def __init__(self, seq_len, hidden_dim, dropout):
        super().__init__()

        input_dim = 2 * seq_len + 4  # dynamic + static

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_dyn, x_stat):
        # x_dyn: (B, L, 2)
        # x_stat: (B, 4)
        x_dyn = x_dyn.reshape(x_dyn.size(0), -1)
        x = torch.cat([x_dyn, x_stat], dim=1)
        return self.net(x)

# ============================================================
# 3. DATASET (Includes measured q)
# ============================================================
class PeltonNARXDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: NARXConfig, stats=None):
        self.seq_len = cfg.seq_len
        self.stride = cfg.stride

        self.dynamic_cols = ["x_n", "q"]
        self.static_cols  = ["T_e", "T_q", "y_r", "h0"]
        self.target_col   = "x3"

        self.samples = []

        for _, g in df.groupby("scenario_id"):
            g = g.sort_values("time").reset_index(drop=True)

            x_dyn = g[self.dynamic_cols].values.astype(np.float32)
            x_stat = g[self.static_cols].iloc[0].values.astype(np.float32)
            y = g[self.target_col].values.astype(np.float32)

            for t in range(self.seq_len, len(g) - 1, self.stride):
                self.samples.append((
                    x_dyn[t-self.seq_len:t],
                    x_stat,
                    y[t+1]   # x3(t+1)
                ))

        if len(self.samples) == 0:
            raise RuntimeError("No valid samples found.")

        # ---------- normalization ----------
        if stats is None:
            dyn = np.concatenate([s[0] for s in self.samples], axis=0)
            stat = np.vstack([s[1] for s in self.samples])
            targ = np.array([s[2] for s in self.samples])

            self.stats = {
                "dyn_mean": dyn.mean(axis=0),
                "dyn_std": dyn.std(axis=0) + 1e-6,
                "stat_mean": stat.mean(axis=0),
                "stat_std": stat.std(axis=0) + 1e-6,
                "y_mean": targ.mean(),
                "y_std": targ.std() + 1e-6,
            }
        else:
            self.stats = stats

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_dyn, x_stat, y = self.samples[idx]

        x_dyn = (x_dyn - self.stats["dyn_mean"]) / self.stats["dyn_std"]
        x_stat = (x_stat - self.stats["stat_mean"]) / self.stats["stat_std"]
        y = (y - self.stats["y_mean"]) / self.stats["y_std"]

        return (
            torch.tensor(x_dyn),
            torch.tensor(x_stat),
            torch.tensor([y])
        )

# ============================================================
# 4. Model fitting loop
# ============================================================
train_losses = []
val_losses = []
test_losses = []

def train_narx():
    global train_losses, val_losses, test_losses
    cfg = NARXConfig()
    train_losses, val_losses, test_losses = [], [], []
    df = pd.read_csv(cfg.csv_path)

    # -------- scenario split --------
    sids = df["scenario_id"].unique()
    np.random.shuffle(sids)

    n = len(sids)
    train_ids = sids[:int(0.7 * n)]
    val_ids   = sids[int(0.7 * n):int(0.85 * n)]
    test_ids  = sids[int(0.85 * n):]

    df_train = df[df["scenario_id"].isin(train_ids)]
    df_val   = df[df["scenario_id"].isin(val_ids)]
    df_test  = df[df["scenario_id"].isin(test_ids)]

    ds_train = PeltonNARXDataset(df_train, cfg)
    ds_val   = PeltonNARXDataset(df_val, cfg, stats=ds_train.stats)
    ds_test  = PeltonNARXDataset(df_test, cfg, stats=ds_train.stats)

    dl_train = DataLoader(ds_train, cfg.batch_size, shuffle=True)
    dl_val   = DataLoader(ds_val, cfg.batch_size)
    dl_test  = DataLoader(ds_test, cfg.batch_size)

    model = NARXPredictor(
        seq_len=cfg.seq_len,
        hidden_dim=cfg.hidden_dim,
        dropout=cfg.dropout
    ).to(cfg.device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    loss_fn = nn.MSELoss()

    best_val = np.inf
    patience = 0
    best_state = None

    for epoch in range(cfg.epochs):
        # ---- train ----
        model.train()
        tr_loss = 0.0

        for x_dyn, x_stat, y in dl_train:
            x_dyn = x_dyn.to(cfg.device)
            x_stat = x_stat.to(cfg.device)
            y = y.to(cfg.device)

            opt.zero_grad()
            y_hat = model(x_dyn, x_stat)
            loss = loss_fn(y_hat, y)
            loss.backward()
            opt.step()

            tr_loss += loss.item()

        tr_loss /= len(dl_train)

        # ---- validation ----
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for x_dyn, x_stat, y in dl_val:
                x_dyn = x_dyn.to(cfg.device)
                x_stat = x_stat.to(cfg.device)
                y = y.to(cfg.device)

                y_hat = model(x_dyn, x_stat)
                val_loss += loss_fn(y_hat, y).item()

        val_loss /= len(dl_val)

        # ---- test (evaluation only) ----
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for x_dyn, x_stat, y in dl_test:
                x_dyn = x_dyn.to(cfg.device)
                x_stat = x_stat.to(cfg.device)
                y = y.to(cfg.device)
                test_loss += loss_fn(model(x_dyn, x_stat), y).item()

        test_loss /= len(dl_test)

        print(
            f"Epoch {epoch+1:03d} | "
            f"train MSE: {tr_loss:.6f} | "
            f"val MSE: {val_loss:.6f}"
        )
        # Store losses for plotting.
        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        test_losses.append(test_loss)


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
class NARXPredictor(nn.Module):
    def __init__(self, seq_len, hidden_dim, dropout):
        super().__init__()
        input_dim = 2 * seq_len + 4
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x_dyn, x_stat):
        x_dyn = x_dyn.reshape(x_dyn.size(0), -1)
        x = torch.cat([x_dyn, x_stat], dim=1)
        return self.net(x)
class PeltonNARXDataset(Dataset):
    def __init__(self, df, seq_len, stride, stats):
        self.seq_len = seq_len
        self.stride = stride
        self.stats = stats

        self.dynamic_cols = ["x_n", "q"]
        self.static_cols = ["T_e", "T_q", "y_r", "h0"]
        self.target_col = "x3"

        self.samples = []

        for _, g in df.groupby("scenario_id"):
            g = g.sort_values("time").reset_index(drop=True)

            x_dyn = g[self.dynamic_cols].values.astype(np.float32)
            x_stat = g[self.static_cols].iloc[0].values.astype(np.float32)
            y = g[self.target_col].values.astype(np.float32)

            for t in range(seq_len, len(g) - 1, stride):
                self.samples.append(
                    (x_dyn[t-seq_len:t], x_stat, y[t+1])
                )

        if len(self.samples) == 0:
            raise RuntimeError("No valid samples found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_dyn, x_stat, y = self.samples[idx]

        x_dyn = (x_dyn - self.stats["dyn_mean"]) / self.stats["dyn_std"]
        x_stat = (x_stat - self.stats["stat_mean"]) / self.stats["stat_std"]
        y = (y - self.stats["y_mean"]) / self.stats["y_std"]

        return (
            torch.tensor(x_dyn),
            torch.tensor(x_stat),
            torch.tensor([y]),
        )
def load_trained_model(ckpt_path):
    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    cfg = ckpt["config"]
    stats = ckpt["stats"]

    model = NARXPredictor(
        seq_len=cfg["seq_len"],
        hidden_dim=cfg["hidden_dim"],
        dropout=cfg["dropout"],
    )
    model.load_state_dict(ckpt["model"])
    model.eval()

    return model, stats, cfg
def test_model(
    ckpt_path,
    csv_path,
    batch_size=256,
    plot_csv_path=None,
):
    # load model
    model, stats, cfg = load_trained_model(ckpt_path)

    # load data
    df = pd.read_csv(csv_path)

    # choose some test scenarios (simple, deterministic)
    scenario_ids = df["scenario_id"].unique()
    test_ids = scenario_ids[int(0.85 * len(scenario_ids)):]  # last 15%
    df_test = df[df["scenario_id"].isin(test_ids)]

    # dataset + loader
    ds_test = PeltonNARXDataset(
        df_test,
        seq_len=cfg["seq_len"],
        stride=cfg["stride"],
        stats=stats,
    )

    dl_test = DataLoader(ds_test, batch_size=batch_size, shuffle=False)

    # run inference
    y_true_n, y_pred_n = [], []

    with torch.no_grad():
        for x_dyn, x_stat, y in dl_test:
            y_hat = model(x_dyn, x_stat)
            y_true_n.append(y.numpy())
            y_pred_n.append(y_hat.numpy())

    y_true_n = np.vstack(y_true_n).squeeze()
    y_pred_n = np.vstack(y_pred_n).squeeze()

    # -------------------------
    # errors (normalized)
    # -------------------------
    mse_n = np.mean((y_pred_n - y_true_n) ** 2)
    rmse_n = np.sqrt(mse_n)
    mae_n = np.mean(np.abs(y_pred_n - y_true_n))

    # -------------------------
    # errors (physical units)
    # -------------------------
    y_std = stats["y_std"]
    y_mean = stats["y_mean"]

    y_true = y_true_n * y_std + y_mean
    y_pred = y_pred_n * y_std + y_mean

    mse = np.mean((y_pred - y_true) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_pred - y_true))

    print("\n=== TEST RESULTS ===")
    print(f"Normalized  MSE : {mse_n:.6f}")
    print(f"Normalized RMSE : {rmse_n:.6f}")
    print(f"Normalized  MAE : {mae_n:.6f}")
    print(f"Physical    MSE : {mse:.6e}")
    print(f"Physical   RMSE : {rmse:.6e}")
    print(f"Physical    MAE : {mae:.6e}")

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, s=5, alpha=0.3)
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], "k--")
    plt.xlabel("True x3(t+1)")
    plt.ylabel("Predicted x3(t+1)")
    plt.title("Predicted vs True (test set)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {
        "mse_norm": mse_n,
        "rmse_norm": rmse_n,
        "mae_norm": mae_n,
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "y_true": y_true,
        "y_pred": y_pred,
    }


if __name__ == "__main__":
    results = test_model(
        ckpt_path="models/pelton-narx-model-plotting.pt",
        csv_path="data/hydraulic-dataset/hydraulic-dataset.csv",
        plot_csv_path=None,
    )
